In [1]:
import os
os.chdir(r"C:\Users\anyaa\Documents\nlp\quora")
import pandas as pd
import numpy as np
from scipy.sparse import hstack,vstack
import contractions
df=pd.read_csv("questions.csv")
df.head(3)
df=df.dropna()
df.isnull().sum()

id              0
qid1            0
qid2            0
question1       0
question2       0
is_duplicate    0
dtype: int64

In [2]:
# Lowercase 
df['question1'] = df['question1'].str.lower()
df['question2'] = df['question2'].str.lower()
# Contractions 
df['question1'] = df['question1'].map(contractions.fix)
df['question2'] = df['question2'].map(contractions.fix)
# Remove special characters 
df['question1'] = df['question1'].str.replace(r"[^a-zA-Z0-9?!\s]", " ", regex=True)
df['question2'] = df['question2'].str.replace(r"[^a-zA-Z0-9?!\s]", " ", regex=True)
# Remove extra spaces 
df['question1'] = df['question1'].str.replace(r"\s+", " ", regex=True).str.strip()
df['question2'] = df['question2'].str.replace(r"\s+", " ", regex=True).str.strip()

In [3]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    df[['question1','question2']],
    df['is_duplicate'],
    test_size=0.2,
    random_state=42
)

In [4]:
import spacy
nlp = spacy.load("en_core_web_sm",disable=["parser","ner"])
def lemmatize_series(series):
    return [
        " ".join([token.lemma_ for token in doc])
        for doc in nlp.pipe(series, batch_size=500)
    ]

X_train['question1'] = lemmatize_series(X_train['question1'])
X_train['question2'] = lemmatize_series(X_train['question2'])

In [7]:
# Word lengths
l1=X_train['question1'].str.split().str.len()
l2= X_train['question2'].str.split().str.len()
X_train['lenq1'] = l1
X_train['lenq2'] =l2
X_train['lendiff']= abs(l2-l1)
#common words
q1_words = X_train['question1'].str.split()
q2_words = X_train['question2'].str.split()

X_train['common_word_count'] = [
    len(set(a) & set(b)) for a,b in zip(q1_words, q2_words)
]
#jaccard similarity
def jaccard(q1, q2):
    w1 = set(str(q1).split())
    w2 = set(str(q2).split())
    if len(w1.union(w2)) == 0:
        return 0
    return len(w1.intersection(w2)) / len(w1.union(w2))

X_train['jaccard'] = [
    len(set(a.split()) & set(b.split())) /
    len(set(a.split()) | set(b.split()))
    if len(set(a.split()) | set(b.split())) > 0 else 0
    for a,b in zip(X_train['question1'], X_train['question2'])
]

def get_qtype(q):
    q = str(q).lower().strip()
    for w in ['what','how','why','who','when','where','which','is','are','can','do']:
        if q.startswith(w):
            return w
    return 'other'

q1_type = X_train['question1'].apply(get_qtype)
q2_type = X_train['question2'].apply(get_qtype)
X_train['same_qtype'] = (q1_type == q2_type).astype(int)


In [8]:
from sklearn.feature_extraction.text import TfidfVectorizer
tfidf = TfidfVectorizer(max_features=20000,ngram_range=(1,2),min_df=2)

tfidf.fit(X_train['question1'].tolist()+X_train['question2'].tolist())
q1 = tfidf.transform(X_train['question1'])
q2 = tfidf.transform(X_train['question2'])

X_tfidf_train = hstack([q1, q2])

train_cosine = q1.multiply(q2).sum(axis=1).A1
X_train['cosine_sim'] = train_cosine

In [9]:
#word to vec
from gensim.models import Word2Vec
sentences = (
    X_train['question1'].str.split().tolist() +
    X_train['question2'].str.split().tolist()
)
w2v = Word2Vec(
    sentences,
    vector_size=100,
    window=5,
    min_count=2,
    workers=4
)
import numpy as np
def sent_vec(text):
    words = text.split()
    vecs = [w2v.wv[w] for w in words if w in w2v.wv]
    
    if len(vecs) == 0:
        return np.zeros(100)
        
    return np.mean(vecs, axis=0)
q1_vec_train = np.vstack(X_train['question1'].apply(sent_vec))
q2_vec_train = np.vstack(X_train['question2'].apply(sent_vec))
w2v_diff_train = q1_vec_train - q2_vec_train
w2v_prod_train = q1_vec_train * q2_vec_train

X_train_w2v = np.hstack([q1_vec_train, q2_vec_train, w2v_diff_train, w2v_prod_train])


from numpy.linalg import norm

w2v_cos_train = (
    np.sum(q1_vec_train * q2_vec_train, axis=1) /
    (norm(q1_vec_train, axis=1) * norm(q2_vec_train, axis=1) + 1e-9)
)

X_train['w2v_cosine'] = w2v_cos_train

In [10]:
from scipy.sparse import csr_matrix
numeric_cols = ['lenq1','lenq2','lendiff','common_word_count','jaccard','cosine_sim','w2v_cosine','same_qtype']
num_train = csr_matrix(X_train[numeric_cols].to_numpy())

X_final_train = hstack([X_tfidf_train, num_train, X_train_w2v])


In [12]:
X_test['question1'] = lemmatize_series(X_test['question1'])
X_test['question2'] = lemmatize_series(X_test['question2'])
    
q1_vec_test = np.vstack(X_test['question1'].apply(sent_vec))
q2_vec_test = np.vstack(X_test['question2'].apply(sent_vec))

w2v_cos_test = (
    np.sum(q1_vec_test * q2_vec_test, axis=1) /
    (norm(q1_vec_test, axis=1) * norm(q2_vec_test, axis=1) + 1e-9)
)
X_test['w2v_cosine'] = w2v_cos_test

w2v_diff_test = q1_vec_test - q2_vec_test
w2v_prod_test = q1_vec_test * q2_vec_test
X_test_w2v = np.hstack([q1_vec_test, q2_vec_test, w2v_diff_test, w2v_prod_test])

q1_test = tfidf.transform(X_test['question1'])
q2_test = tfidf.transform(X_test['question2'])

l1=X_test['question1'].str.split().str.len()
l2= X_test['question2'].str.split().str.len()
X_test['lenq1'] = l1
X_test['lenq2'] =l2
X_test['lendiff']= abs(l2-l1)
def common_words(q1, q2):
    w1 = set(str(q1).lower().split())
    w2 = set(str(q2).lower().split())
    return len(w1.intersection(w2))
X_test['common_word_count'] = [
    len(set(a.split()) & set(b.split()))
    for a,b in zip(X_test['question1'], X_test['question2'])
]

def jaccard(q1, q2):
    w1 = set(str(q1).lower().split())
    w2 = set(str(q2).lower().split())
    if len(w1.union(w2)) == 0:
        return 0
    return len(w1.intersection(w2)) / len(w1.union(w2))
X_test['jaccard'] = X_test.apply(
    lambda row: jaccard(row['question1'], row['question2']),
    axis=1
)
test_cosine = q1_test.multiply(q2_test).sum(axis=1).A1
X_test['cosine_sim'] = test_cosine

q1_type = X_test['question1'].apply(get_qtype)
q2_type = X_test['question2'].apply(get_qtype)
X_test['same_qtype'] = (q1_type == q2_type).astype(int)

numeric_cols = [
    'lenq1',
    'lenq2',
    'lendiff',
    'common_word_count',
    'jaccard',
    'cosine_sim',
    'w2v_cosine',
    'same_qtype'
]
from scipy.sparse import csr_matrix

num_test_sparse = csr_matrix(X_test[numeric_cols].values)
from scipy.sparse import hstack

X_tfidf_test = hstack([q1_test, q2_test])
X_final_test = hstack([X_tfidf_test, num_test_sparse,X_test_w2v])


In [15]:
from lightgbm import LGBMClassifier, early_stopping
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report

# 1. Split TRAIN into train + validation
X_tr, X_val, y_tr, y_val = train_test_split(
    X_final_train, y_train,
    test_size=0.2,
    stratify=y_train,   # keeps class balance
    random_state=42
)


neg = (y_tr == 0).sum()
pos = (y_tr == 1).sum()

model = LGBMClassifier(
    n_estimators=1000,
    learning_rate=0.05,
    num_leaves=63,
    colsample_bytree=0.8,
    subsample=0.8,
    subsample_freq=1,
    reg_alpha=0.1,
    reg_lambda=0.1,
    scale_pos_weight=neg / pos,
    random_state=42,
)

# 3. Train with early stopping on VALIDATION set
model.fit(
    X_tr, y_tr,
    eval_set=[(X_val, y_val)],
    eval_metric='binary_logloss',
    callbacks=[early_stopping(50)]
)

# 4. Final evaluation ONLY on test set
y_pred = model.predict(X_final_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("F1:", f1_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

[LightGBM] [Info] Number of positive: 95741, number of negative: 163041
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 36.088153 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1075469
[LightGBM] [Info] Number of data points in the train set: 258782, number of used features: 37513
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.369968 -> initscore=-0.532355
[LightGBM] [Info] Start training from score -0.532355
Training until validation scores don't improve for 50 rounds
Did not meet early stopping. Best iteration is:
[1000]	valid_0's binary_logloss: 0.375933


C:\ProgramData\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\Users\anyaa\AppData\Roaming\Python\Python313\site-packages\lightgbm\basic.py:1238: UserWarning: Converting data to scipy sparse matrix.
  _log_warning("Converting data to scipy sparse matrix.")


Accuracy: 0.8189316186472115
F1: 0.7750587584681322
              precision    recall  f1-score   support

           0       0.90      0.80      0.85     51240
           1       0.71      0.85      0.78     29630

    accuracy                           0.82     80870
   macro avg       0.81      0.83      0.81     80870
weighted avg       0.83      0.82      0.82     80870



In [21]:
from sentence_transformers import SentenceTransformer
import numpy as np

sbert = SentenceTransformer('all-MiniLM-L6-v2')

# Test on small sample first to make sure it works
sample_q1 = X_train['question1'].tolist()[:100]
sample_q2 = X_train['question2'].tolist()[:100]
test_enc = sbert.encode(sample_q1, batch_size=64)
print("Sample encoding shape:", test_enc.shape)  # should be (100, 384)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Sample encoding shape: (100, 384)


In [22]:
print("Encoding Q1...")
q1_sbert_train = sbert.encode(
    X_train['question1'].tolist(),
    batch_size=256,
    show_progress_bar=True
)

print("Encoding Q2...")
q2_sbert_train = sbert.encode(
    X_train['question2'].tolist(),
    batch_size=256,
    show_progress_bar=True
)

# Do the same for test set
print("Encoding Q1 test...")
q1_sbert_test = sbert.encode(
    X_test['question1'].tolist(),
    batch_size=256,
    show_progress_bar=True
)

print("Encoding Q2 test...")
q2_sbert_test = sbert.encode(
    X_test['question2'].tolist(),
    batch_size=256,
    show_progress_bar=True
)

print("Train shape:", q1_sbert_train.shape)  # (400000, 384)

Encoding Q1...


Batches:   0%|          | 0/1264 [00:00<?, ?it/s]

Encoding Q2...


Batches:   0%|          | 0/1264 [00:00<?, ?it/s]

Encoding Q1 test...


Batches:   0%|          | 0/316 [00:00<?, ?it/s]

Encoding Q2 test...


Batches:   0%|          | 0/316 [00:00<?, ?it/s]

Train shape: (323478, 384)


In [23]:
from numpy.linalg import norm

# Train
sbert_cosine_train = (
    np.sum(q1_sbert_train * q2_sbert_train, axis=1) /
    (norm(q1_sbert_train, axis=1) * norm(q2_sbert_train, axis=1) + 1e-9)
)
X_train['sbert_cosine'] = sbert_cosine_train

# Test
sbert_cosine_test = (
    np.sum(q1_sbert_test * q2_sbert_test, axis=1) /
    (norm(q1_sbert_test, axis=1) * norm(q2_sbert_test, axis=1) + 1e-9)
)
X_test['sbert_cosine'] = sbert_cosine_test

In [24]:
# Stack Q1 and Q2 embeddings side by side
X_sbert_train = np.hstack([q1_sbert_train, q2_sbert_train])  # (n, 768)
X_sbert_test  = np.hstack([q1_sbert_test,  q2_sbert_test])   # (n, 768)

print("SBERT train matrix shape:", X_sbert_train.shape)

SBERT train matrix shape: (323478, 768)


In [26]:
# Remove w2v_cosine, keep everything else
handcrafted_cols = [
    'lendiff',
    'lenq1', 'lenq2',
    'common_word_count',
    'jaccard',
    'cosine_sim',      
    'sbert_cosine'     # new 
]

handcrafted_train = X_train[handcrafted_cols].values
handcrafted_test  = X_test[handcrafted_cols].values

In [37]:
from scipy.sparse import hstack, csr_matrix
import numpy as np

# SBERT difference and product vectors
sbert_diff_train = q1_sbert_train - q2_sbert_train
sbert_prod_train = q1_sbert_train * q2_sbert_train

sbert_diff_test = q1_sbert_test - q2_sbert_test
sbert_prod_test = q1_sbert_test * q2_sbert_test

# Handcrafted cols — keep sbert_cosine
handcrafted_cols = [
    'lendiff', 'lenq1', 'lenq2',
    'common_word_count', 'jaccard',
    'cosine_sim',
    'sbert_cosine'
]

handcrafted_train = X_train[handcrafted_cols].values
handcrafted_test  = X_test[handcrafted_cols].values

# Final matrix
X_final_train = hstack([
    X_tfidf_train,                        # 40,000 dims
    csr_matrix(q1_sbert_train),           # 384 dims
    csr_matrix(q2_sbert_train),           # 384 dims
    csr_matrix(sbert_diff_train),         # 384 dims ← new
    csr_matrix(sbert_prod_train),         # 384 dims ← new
    csr_matrix(handcrafted_train)         # 8 dims
])

X_final_test = hstack([
    X_tfidf_test,
    csr_matrix(q1_sbert_test),
    csr_matrix(q2_sbert_test),
    csr_matrix(sbert_diff_test),
    csr_matrix(sbert_prod_test),
    csr_matrix(handcrafted_test)
])

print("Final shape:", X_final_train.shape)
# Should be around (n, 41544)

Final shape: (323478, 41543)


In [38]:
from lightgbm import LGBMClassifier
from sklearn.metrics import classification_report, accuracy_score, f1_score

model = LGBMClassifier(
    n_estimators=300,
    learning_rate=0.1,
    num_leaves=31,
    colsample_bytree=0.8,
    subsample=0.8,
    scale_pos_weight=neg/pos,
    random_state=42,
)

model.fit(X_final_train, y_train)
y_pred = model.predict(X_final_test)

print(f"Accuracy: {accuracy_score(y_test, y_pred)}")
print(f"F1: {f1_score(y_test, y_pred)}")
print(classification_report(y_test, y_pred))

[LightGBM] [Info] Number of positive: 119676, number of negative: 203802
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 68.584590 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1426292
[LightGBM] [Info] Number of data points in the train set: 323478, number of used features: 40808
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.369966 -> initscore=-0.532361
[LightGBM] [Info] Start training from score -0.532361


C:\ProgramData\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Accuracy: 0.8557808828984791
F1: 0.8198625376476948
              precision    recall  f1-score   support

           0       0.93      0.83      0.88     51240
           1       0.76      0.90      0.82     29630

    accuracy                           0.86     80870
   macro avg       0.84      0.86      0.85     80870
weighted avg       0.87      0.86      0.86     80870



In [42]:
import joblib

# Save the model
joblib.dump(model, 'model.pkl')

# Save the tfidf vectorizer
joblib.dump(tfidf, 'tfidf.pkl')

# Save the sbert model name (no need to save, just reload)
# Save the w2v model
w2v.save('w2v.model')

print("All saved!")

All saved!


In [44]:
def predict_duplicate(q1, q2):
    # --- Lemmatize ---
    q1_lem = " ".join([token.lemma_ for token in nlp(q1)])
    q2_lem = " ".join([token.lemma_ for token in nlp(q2)])

    # --- Handcrafted features ---
    w1 = set(q1_lem.split())
    w2 = set(q2_lem.split())
    lenq1 = len(q1_lem.split())
    lenq2 = len(q2_lem.split())
    lendiff = abs(lenq1 - lenq2)
    common = len(w1 & w2)
    jaccard = len(w1 & w2) / len(w1 | w2) if len(w1 | w2) > 0 else 0

    # --- TF-IDF ---
    q1_tfidf = tfidf.transform([q1_lem])
    q2_tfidf = tfidf.transform([q2_lem])
    tfidf_hstack = hstack([q1_tfidf, q2_tfidf])
    cosine_sim = float(q1_tfidf.multiply(q2_tfidf).sum(axis=1).A1[0])

    # --- SBERT ---
    q1_sbert = sbert.encode([q1_lem])        # (1, 384)
    q2_sbert = sbert.encode([q2_lem])        # (1, 384)
    sbert_diff = q1_sbert - q2_sbert         # (1, 384)
    sbert_prod = q1_sbert * q2_sbert         # (1, 384)
    sbert_cos = float(
        np.sum(q1_sbert * q2_sbert) /
        (norm(q1_sbert) * norm(q2_sbert) + 1e-9)
    )

    # --- Handcrafted array ---
    handcrafted = np.array([[
        lendiff, lenq1, lenq2,
        common, jaccard, cosine_sim,
        sbert_cos
    ]])

    # --- Final matrix (must match 41,544) ---
    X = hstack([
        tfidf_hstack,              # 40,000
        csr_matrix(q1_sbert),     # 384
        csr_matrix(q2_sbert),     # 384
        csr_matrix(sbert_diff),   # 384
        csr_matrix(sbert_prod),   # 384
        csr_matrix(handcrafted)   # 8
    ])                             # total: 41,544 ✅

    # --- Predict ---
    pred = model.predict(X)[0]
    prob = model.predict_proba(X)[0][1]

    print(f"\nQuestion 1: {q1}")
    print(f"Question 2: {q2}")
    print(f"Prediction : {'✅ DUPLICATE' if pred == 1 else '❌ NOT DUPLICATE'}")
    print(f"Confidence : {prob:.1%}")
    print("-" * 50)

In [54]:
# Try some examples
predict_duplicate(
    "What is the best programming language to learn?",
    "Which programming language should a beginner learn first?"
)

predict_duplicate(
    "What is the capital of France?",
    "How do I make pasta?"
)

predict_duplicate(
    "What are the best ways to lose weight?",
    "How can I reduce my body weight effectively?"
)

C:\ProgramData\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(



Question 1: What is the best programming language to learn?
Question 2: Which programming language should a beginner learn first?
Prediction : ✅ DUPLICATE
Confidence : 93.9%
--------------------------------------------------


C:\ProgramData\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(



Question 1: What is the capital of France?
Question 2: How do I make pasta?
Prediction : ❌ NOT DUPLICATE
Confidence : 0.0%
--------------------------------------------------

Question 1: What are the best ways to lose weight?
Question 2: How can I reduce my body weight effectively?
Prediction : ✅ DUPLICATE
Confidence : 91.3%
--------------------------------------------------


C:\ProgramData\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
